# Inspect the SAE concept-manifold prompts

This one-off notebook reads the saved **prompts and ground-truth labels** from
`dalg-cache/sae_manifold_prompts/`. It does not regenerate data or load a language model.

The eight builders follow [Goodfire's pinned data.py](https://github.com/goodfire-ai/sae-manifold/blob/f2632ddbb25f4c670254ceea999058b9ba4b0450/data.py),
including their sampling strides, filters, ordering, and caps. The only source
compatibility adjustment is using `Salesforce/wikitext` instead of its short alias.

Things to inspect:
- Age prompts retain their trailing space; `repr(prompt)` below makes it visible.
- Years are **1800–1998**, matching the code's exclusive stop, despite the upstream docstring.
- `sent_length.labels.n_tokens` is a **whitespace-separated word count**, not a model token count.
  Its sentence splitting and punctuation are preserved exactly from upstream.
- These plots show **label coordinates**, not measured activation geometry.

To rebuild into a new directory, run from the repository root:
```bash
PYTHONPATH=src .venv/bin/python -m dalg.data.build_sae_manifold_prompts --out dalg-cache/sae_manifold_prompts_new
```

In [1]:
from pathlib import Path
import json
from html import escape

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display

pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 50)

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / "dalg-cache/sae_manifold_prompts"

metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
with (DATA_DIR / metadata["data_file"]).open(encoding="utf-8") as handle:
    records = [json.loads(line) for line in handle]
df = pd.DataFrame.from_records(records)
ORDER = metadata["manifolds"]

# Reload checks keep row identity and per-manifold counts tied to the saved metadata.
assert df.row_id.tolist() == list(range(len(df)))
assert len(df) == metadata["total_rows"]
assert df.groupby("subset", sort=False).size().to_dict() == metadata["counts"]
assert df.prompt.map(lambda value: isinstance(value, str)).all()
assert df.labels.map(lambda value: isinstance(value, dict)).all()

df["prompt_chars"] = df.prompt.str.len()
df["prompt_words"] = df.prompt.map(lambda text: len(text.split()))
df["edge_whitespace"] = df.prompt.ne(df.prompt.str.strip())
frames = {}
for name in ORDER:
    rows = df.loc[df.subset.eq(name)].reset_index(drop=True)
    frames[name] = pd.concat([rows.drop(columns="labels"), pd.json_normalize(rows.labels)], axis=1)

print(f"Loaded {len(df):,} prompts across {len(ORDER)} manifolds from {DATA_DIR}")
print("Upstream revision:", metadata["upstream"]["revision"])
print("Source settings are available in metadata['sources'].")

Loaded 11,382 prompts across 8 manifolds from /orfeo/cephfs/home/dssc/zenocosini/decomposing-activations-local-geometry/dalg-cache/sae_manifold_prompts
Upstream revision: f2632ddbb25f4c670254ceea999058b9ba4b0450
Source settings are available in metadata['sources'].


## Counts, lengths, and duplicates

Duplicate counts mean repeated exact prompt strings **within each manifold**, after
its upstream selection rules. They are reported without removing any rows.

In [2]:
summary_rows = []
for name in ORDER:
    part = frames[name]
    summary_rows.append({
        "subset": name,
        "rows": len(part),
        "unique_prompts": part.prompt.nunique(),
        "duplicate_prompts": int(part.prompt.duplicated().sum()),
        "empty_prompts": int(part.prompt.str.strip().eq("").sum()),
        "edge_whitespace": int(part.edge_whitespace.sum()),
        "chars_min": part.prompt_chars.min(),
        "chars_mean": round(part.prompt_chars.mean(), 1),
        "chars_max": part.prompt_chars.max(),
        "words_min": part.prompt_words.min(),
        "words_mean": round(part.prompt_words.mean(), 1),
        "words_max": part.prompt_words.max(),
    })
summary = pd.DataFrame(summary_rows).set_index("subset")
display(summary)
px.box(df, x="subset", y="prompt_words", points=False,
       category_orders={"subset": ORDER}, height=350,
       labels={"prompt_words": "Whitespace word count", "subset": "Manifold"},
       title="Prompt lengths by manifold").show()

,rows,unique_prompts,duplicate_prompts,empty_prompts,edge_whitespace,chars_min,chars_mean,chars_max,words_min,words_mean,words_max
subset,,,,,,,,,,,
colors,1850,1850,0,0,0,37,37.0,37,8,8.0,8
age,99,99,0,0,99,22,22.9,23,5,5.0,5
temperature,150,150,0,0,0,39,40.2,41,6,6.0,6
geography,3028,3024,4,0,0,82,93.7,98,13,13.0,13
days,56,56,0,0,0,19,25.5,35,4,4.6,7
years,199,199,0,0,0,16,16.0,16,4,4.0,4
formality,1000,1000,0,0,0,4,88.2,418,1,15.1,81
sent_length,5000,5000,0,0,0,10,205.0,557,3,38.6,80


## Representative prompts from every manifold

The first, middle, and last rows show ordering and boundary examples. Prompt
representations preserve visible whitespace; labels are the original dictionaries.

In [3]:
example_rows = []
for name in ORDER:
    part = df.loc[df.subset.eq(name)]
    for position in (0, len(part) // 2, len(part) - 1):
        row = part.iloc[position]
        example_rows.append({"subset": name, "row_id": row.row_id,
                             "prompt_repr": repr(row.prompt), "labels": row.labels})
display(pd.DataFrame(example_rows))

,subset,row_id,prompt_repr,labels
0,colors,0,'The hex code #0e277d is for the color',"{'hex': '#0e277d', 'hue': 0.6291291291291291, 'lightness': 0.2725490196078431, 'saturation': 0.7985611510791367}"
1,colors,925,'The hex code #8066d0 is for the color',"{'hex': '#8066d0', 'hue': 0.7075471698113208, 'lightness': 0.607843137254902, 'saturation': 0.5299999999999999}"
2,colors,1849,'The hex code #f0dd80 is for the color',"{'hex': '#f0dd80', 'hue': 0.13839285714285718, 'lightness': 0.7215686274509804, 'saturation': 0.7887323943661971}"
3,age,1850,'They are 1 years old. ',{'age': 1.0}
4,age,1899,'They are 50 years old. ',{'age': 50.0}
5,age,1948,'They are 99 years old. ',{'age': 99.0}
6,temperature,1949,"""Today it's -30 degrees Fahrenheit outside""","{'fahrenheit': -30.0, 'celsius': -34.44444444444444}"
7,temperature,2024,"""Today it's 45 degrees Fahrenheit outside""","{'fahrenheit': 45.0, 'celsius': 7.222222222222222}"
8,temperature,2098,"""Today it's 119 degrees Fahrenheit outside""","{'fahrenheit': 119.0, 'celsius': 48.333333333333336}"
9,geography,2099,'The geographical coordinates latitude 34.5166667 and longitude 69.1833344 are in the country of',"{'latitude': 34.5166667, 'longitude': 69.1833344, 'country': 'Afghanistan', 'region': 'Southern Asia', 'continent': 'Asia'}"


## Select a manifold for closer inspection

Edit `MANIFOLD` and rerun this cell. `selected` contains every original record for
that manifold; `selected_labels` is its label table. Printed prompts are untruncated.

In [9]:
MANIFOLD = "days"  # colors, age, temperature, geography, days, years, formality, sent_length
N_EXAMPLES = 5

selected = df.loc[df.subset.eq(MANIFOLD)].copy()
selected_labels = pd.json_normalize(selected.labels.tolist())
print(f"{MANIFOLD}: {len(selected):,} rows; label fields: {selected_labels.columns.tolist()}")
display(selected_labels.describe(include="all"))
for row in selected.sample(n=min(N_EXAMPLES, len(selected)), random_state=0).itertuples():
    print(f"\nrow_id={row.row_id}  prompt={row.prompt!r}")
    print(json.dumps(row.labels, ensure_ascii=False))

# To inspect a specific saved record without changing its text:
# records[123]

days: 56 rows; label fields: ['day', 'day_idx', 'time', 'time_idx']


,day,day_idx,time,time_idx
count,56,56.0000,56,56.000000
unique,7,NaN,8,NaN
top,Monday,NaN,the crack of dawn,NaN
freq,8,NaN,7,NaN
mean,NaN,3.0000,NaN,3.500000
std,NaN,2.0181,NaN,2.312024
min,NaN,0.0000,NaN,0.000000
25%,NaN,1.0000,NaN,1.750000
50%,NaN,3.0000,NaN,3.500000
75%,NaN,5.0000,NaN,5.250000



row_id=5172  prompt="It's evening on Saturday"
{"day": "Saturday", "day_idx": 5, "time": "evening", "time_idx": 5}

row_id=5160  prompt="It's early morning on Friday"
{"day": "Friday", "day_idx": 4, "time": "early morning", "time_idx": 1}

row_id=5167  prompt="It's the crack of dawn on Saturday"
{"day": "Saturday", "day_idx": 5, "time": "the crack of dawn", "time_idx": 0}

row_id=5153  prompt="It's late morning on Thursday"
{"day": "Thursday", "day_idx": 3, "time": "late morning", "time_idx": 2}

row_id=5138  prompt="It's noon on Tuesday"
{"day": "Tuesday", "day_idx": 1, "time": "noon", "time_idx": 3}


## Label completeness and numeric ranges

Missing values are counted within each manifold's own label fields. Numeric-looking
coordinate values are converted only for summaries and plots; saved labels stay unchanged.

In [5]:
label_rows = []
for name in ORDER:
    labels = pd.json_normalize(df.loc[df.subset.eq(name), "labels"].tolist())
    for field in labels:
        values = labels[field]
        missing = values.isna() | values.map(lambda value: isinstance(value, str) and not value.strip())
        numeric = pd.to_numeric(values, errors="coerce")
        is_numeric = bool((~missing).any() and numeric[~missing].notna().all())
        label_rows.append({
            "subset": name, "field": field, "missing": int(missing.sum()),
            "unique": values.nunique(), "numeric": is_numeric,
            "min": numeric.min() if is_numeric else None,
            "max": numeric.max() if is_numeric else None,
        })
label_audit = pd.DataFrame(label_rows)
display(label_audit)
print("Total missing label values:", label_audit.missing.sum())

,subset,field,missing,unique,numeric,min,max
0,colors,hex,0,1850,False,NaN,NaN
1,colors,hue,0,1555,True,0.000000,0.998106
2,colors,lightness,0,253,True,0.200000,0.800000
3,colors,saturation,0,401,True,0.200000,0.800000
4,age,age,0,99,True,1.000000,99.000000
5,temperature,fahrenheit,0,150,True,-30.000000,119.000000
6,temperature,celsius,0,150,True,-34.444444,48.333333
7,geography,latitude,0,2759,True,-46.600000,66.450000
8,geography,longitude,0,2844,True,-175.199997,179.383331
9,geography,country,0,130,False,NaN,NaN


Total missing label values: 0


## Colors: swatches and HLS coordinates

The swatches are a fixed sample. The interactive plot includes every saved color;
hover to see its hex code and row ID.

In [6]:
colors = frames["colors"]
swatches = colors.sample(n=min(36, len(colors)), random_state=0).sort_values("hue")
blocks = []
for color in swatches["hex"]:
    safe_color = escape(color, quote=True)
    blocks.append(f'<div style="text-align:center;font-family:monospace;font-size:12px">'
                  f'<div style="background:{safe_color};height:36px;border:1px solid #aaa"></div>'
                  f'{safe_color}</div>')
display(HTML('<div style="display:grid;grid-template-columns:repeat(12,1fr);gap:6px">'
             + ''.join(blocks) + '</div>'))
fig = go.Figure(go.Scatter3d(
    x=colors.hue, y=colors.lightness, z=colors.saturation,
    mode="markers", marker={"size": 3, "color": colors["hex"].tolist(), "opacity": 0.8},
    text=[f"row_id={row_id}  hex={color}" for row_id, color in zip(colors.row_id, colors["hex"])],
    hovertemplate="%{text}<br>hue=%{x:.3f}<br>lightness=%{y:.3f}<br>saturation=%{z:.3f}<extra></extra>",
))
fig.update_layout(title="Saved colors in HLS label coordinates", height=450,
                  scene={"xaxis_title": "Hue", "yaxis_title": "Lightness", "zaxis_title": "Saturation"},
                  margin={"l": 0, "r": 0, "b": 0, "t": 45})
fig.show()

## Geography and the day/time grid

The geographic view uses a plain longitude/latitude scatter so it needs no external
map service. The grid should contain one prompt for each of the 56 day/time pairs.

In [7]:
geography = frames["geography"].copy()
for field in ("latitude", "longitude"):
    geography[field] = pd.to_numeric(geography[field])
fig = px.scatter(geography, x="longitude", y="latitude", color="continent",
                 hover_data=["row_id", "country", "region"], opacity=0.65,
                 title="Geographic coordinates by continent", height=420)
fig.update_traces(marker_size=5)
fig.update_xaxes(range=[-180, 180])
fig.update_yaxes(range=[-90, 90])
fig.show()
display(geography.groupby("continent", dropna=False).size().rename("rows").to_frame())

days = frames["days"]
day_order = days.sort_values("day_idx").day.drop_duplicates().tolist()
time_order = days.sort_values("time_idx")["time"].drop_duplicates().tolist()
grid = pd.crosstab(days.day, days["time"]).reindex(index=day_order, columns=time_order, fill_value=0)
fig = px.imshow(grid, text_auto=True, aspect="auto", zmin=0, zmax=1,
                color_continuous_scale="Blues", height=350,
                labels={"x": "Time of day", "y": "Day", "color": "Prompts"},
                title="Day × time coverage")
fig.update_layout(coloraxis_showscale=False)
fig.show()

,rows
continent,
Africa,610
Asia,730
Europe,947
North America,370
Oceania,130
South America,241


## Remaining label distributions

Each histogram uses all saved rows. The final panel checks that the upstream
`n_tokens` label agrees with the whitespace word count of the saved prompt.

In [8]:
panels = [
    ("age", "age", "Age"),
    ("temperature", "fahrenheit", "Temperature (Fahrenheit)"),
    ("years", "year", "Year"),
    ("formality", "formality", "Formality score"),
    ("sent_length", "n_tokens", "Sentence length (whitespace words)"),
    ("sent_length", "n_chars", "Sentence length (characters)"),
]
fig = make_subplots(rows=2, cols=3, subplot_titles=[title for _, _, title in panels],
                    vertical_spacing=0.22)
for i, (name, field, title) in enumerate(panels):
    row, col = divmod(i, 3)
    fig.add_trace(go.Histogram(x=frames[name][field], nbinsx=35, name=title), row=row+1, col=col+1)
    fig.update_xaxes(title_text=field, row=row+1, col=col+1)
    fig.update_yaxes(title_text="Rows", row=row+1, col=col+1)
fig.update_layout(height=560, showlegend=False, title="Ground-truth label distributions")
fig.show()

sentences = frames["sent_length"]
print("Word-count label mismatches:", int(sentences.n_tokens.ne(sentences.prompt_words).sum()))
print("Character-count label mismatches:", int(sentences.n_chars.ne(sentences.prompt_chars).sum()))
print("Age prompts with trailing whitespace:", int(frames["age"].edge_whitespace.sum()))
print("Year range:", int(frames["years"].year.min()), "to", int(frames["years"].year.max()))

Word-count label mismatches: 0
Character-count label mismatches: 0
Age prompts with trailing whitespace: 99
Year range: 1800 to 1998
